In [1]:
# ============================================================
# CELL 1 -- NOTEBOOK 6 — S-XGBOOST
# SHAP-PRUNED XGBOOST
# ============================================================

import pandas as pd
import numpy as np
import xgboost as xgb
import joblib

from pathlib import Path

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("=" * 70)
print("NOTEBOOK 6 — S-XGBOOST")
print("=" * 70)

print(f"XGBoost version: {xgb.__version__}")

# Project directories
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
FIELD_DIR = DATA_DIR / "field"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"
FIGURE_DIR = RESULTS_DIR / "figures"
TABLE_DIR = RESULTS_DIR / "tables"
REPORT_DIR = RESULTS_DIR / "reports"
# Create output directories
for folder in [FIGURE_DIR, TABLE_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)


NOTEBOOK 6 — S-XGBOOST
XGBoost version: 3.3.0


In [2]:
# ============================================================
# CELL 2--LOAD SHAP-PRUNED DATASETS
# ============================================================

train_binary_shap = pd.read_csv(
    PROCESSED_DIR / "train_binary_shap.csv"
)

test_binary_shap = pd.read_csv(
    PROCESSED_DIR / "test_binary_shap.csv"
)

print("=" * 70)
print("SHAP-PRUNED DATASETS")
print("=" * 70)

print(f"Training shape: {train_binary_shap.shape}")
print(f"Testing shape:  {test_binary_shap.shape}")

SHAP-PRUNED DATASETS
Training shape: (468, 50)
Testing shape:  (117, 50)


In [3]:
#Cell 4 — Separate predictors and target
# ============================================================
# SEPARATE FEATURES AND TARGET
# ============================================================

TARGET = "RISK_BINARY"

X_train_shap = train_binary_shap.drop(
    columns=[TARGET]
)

y_train = train_binary_shap[TARGET].copy()

X_test_shap = test_binary_shap.drop(
    columns=[TARGET]
)

y_test = test_binary_shap[TARGET].copy()

print("=" * 70)
print("FEATURE / TARGET SEPARATION")
print("=" * 70)

print(f"X_train_shap: {X_train_shap.shape}")
print(f"y_train:      {y_train.shape}")

print(f"X_test_shap:  {X_test_shap.shape}")
print(f"y_test:       {y_test.shape}")

FEATURE / TARGET SEPARATION
X_train_shap: (468, 49)
y_train:      (468,)
X_test_shap:  (117, 49)
y_test:       (117,)


In [4]:
# ============================================================
# CELL 5 --SHAP FEATURE-SPACE VALIDATION
# ============================================================

selected_features_path = (
    RESULTS_DIR / "shap_selected_features_binary.csv"
)

selected_features_df = pd.read_csv(
    selected_features_path
)

selected_features = (
    selected_features_df["Feature"]
    .tolist()
)

print("=" * 70)
print("SHAP FEATURE-SPACE VALIDATION")
print("=" * 70)

print(f"Expected SHAP features: {len(selected_features)}")
print(f"Actual training features: {X_train_shap.shape[1]}")

assert len(selected_features) == 49

assert list(
    X_train_shap.columns
) == selected_features

assert list(
    X_test_shap.columns
) == selected_features

print("✓ Exactly 49 SHAP-selected predictors")
print("✓ Training feature order matches SHAP provenance")
print("✓ Testing feature order matches SHAP provenance")
print("\n✓ SHAP FEATURE SPACE VALIDATED")

SHAP FEATURE-SPACE VALIDATION
Expected SHAP features: 49
Actual training features: 49
✓ Exactly 49 SHAP-selected predictors
✓ Training feature order matches SHAP provenance
✓ Testing feature order matches SHAP provenance

✓ SHAP FEATURE SPACE VALIDATED


In [5]:
# ============================================================
# CELL 6- MODELLING DATA VALIDATION
# ============================================================

print("=" * 70)
print("S-XGBOOST DATA VALIDATION")
print("=" * 70)

assert X_train_shap.shape == (468, 49)
assert X_test_shap.shape == (117, 49)

assert y_train.shape == (468,)
assert y_test.shape == (117,)

assert X_train_shap.isnull().sum().sum() == 0
assert X_test_shap.isnull().sum().sum() == 0

assert X_train_shap.select_dtypes(
    exclude=np.number
).shape[1] == 0

assert X_test_shap.select_dtypes(
    exclude=np.number
).shape[1] == 0

assert sorted(y_train.unique()) == [0, 1]
assert sorted(y_test.unique()) == [0, 1]

print("✓ Training: 468 × 49")
print("✓ Testing: 117 × 49")
print("✓ No missing predictor values")
print("✓ All predictors numeric")
print("✓ Binary target validated")

print("\n✓ S-XGBOOST DATA VALIDATION PASSED")

S-XGBOOST DATA VALIDATION
✓ Training: 468 × 49
✓ Testing: 117 × 49
✓ No missing predictor values
✓ All predictors numeric
✓ Binary target validated

✓ S-XGBOOST DATA VALIDATION PASSED


In [6]:
#Cell 7 — Lock identical baseline parameters
#This is the most important cell for the ablation study.
# ============================================================
# CELL 7- S-XGBOOST — SAME PARAMETERS AS BASELINE
# ============================================================

S_XGBOOST_PARAMS = {
    "objective": "binary:logistic",
    "eval_metric": "logloss",
    "random_state": 42,
    "n_estimators": 100,
    "max_depth": 6,
    "learning_rate": 0.3,
    "subsample": 1.0,
    "colsample_bytree": 1.0
}

print("=" * 70)
print("S-XGBOOST CONFIGURATION")
print("=" * 70)

for parameter, value in S_XGBOOST_PARAMS.items():
    print(f"{parameter:20s}: {value}")

print("\n✓ Parameters identical to Baseline XGBoost")
print("✓ Only feature space has changed")
print("✓ No hyperparameter tuning")
print("✓ No resampling")
print("✓ No class weighting")

S-XGBOOST CONFIGURATION
objective           : binary:logistic
eval_metric         : logloss
random_state        : 42
n_estimators        : 100
max_depth           : 6
learning_rate       : 0.3
subsample           : 1.0
colsample_bytree    : 1.0

✓ Parameters identical to Baseline XGBoost
✓ Only feature space has changed
✓ No hyperparameter tuning
✓ No resampling
✓ No class weighting


In [7]:
# ============================================================
# CELL 8- TRAIN S-XGBOOST
# ============================================================

s_xgboost = xgb.XGBClassifier(
    **S_XGBOOST_PARAMS
)

s_xgboost.fit(
    X_train_shap,
    y_train
)

print("=" * 70)
print("S-XGBOOST TRAINING COMPLETE")
print("=" * 70)

print("✓ Model trained")
print("✓ Training observations: 468")
print("✓ SHAP-selected predictors: 49")
print("✓ No test data used during training")

S-XGBOOST TRAINING COMPLETE
✓ Model trained
✓ Training observations: 468
✓ SHAP-selected predictors: 49
✓ No test data used during training


In [8]:
# ============================================================
# CELL 9 -S-XGBOOST PREDICTIONS
# ============================================================

y_pred_s = s_xgboost.predict(
    X_test_shap
)

y_prob_s = s_xgboost.predict_proba(
    X_test_shap
)[:, 1]

print("=" * 70)
print("S-XGBOOST PREDICTIONS")
print("=" * 70)

print(f"Predicted classes: {len(y_pred_s)}")
print(f"Predicted probabilities: {len(y_prob_s)}")

assert len(y_pred_s) == 117
assert len(y_prob_s) == 117

print("✓ Prediction count matches test set")

S-XGBOOST PREDICTIONS
Predicted classes: 117
Predicted probabilities: 117
✓ Prediction count matches test set


In [9]:
#Cell 10 — Calculate S-XGBoost metrics
# ============================================================
# S-XGBOOST PERFORMANCE METRICS
# ============================================================

roc_auc_s = roc_auc_score(
    y_test,
    y_prob_s
)

pr_auc_s = average_precision_score(
    y_test,
    y_prob_s
)

accuracy_s = accuracy_score(
    y_test,
    y_pred_s
)

precision_s = precision_score(
    y_test,
    y_pred_s,
    zero_division=0
)

recall_s = recall_score(
    y_test,
    y_pred_s,
    zero_division=0
)

f1_s = f1_score(
    y_test,
    y_pred_s,
    zero_division=0
)

cm_s = confusion_matrix(
    y_test,
    y_pred_s
)

tn_s, fp_s, fn_s, tp_s = cm_s.ravel()

specificity_s = tn_s / (
    tn_s + fp_s
)

print("=" * 70)
print("S-XGBOOST PERFORMANCE")
print("=" * 70)

print(f"ROC-AUC:       {roc_auc_s:.4f}")
print(f"PR-AUC:        {pr_auc_s:.4f}")
print(f"F1-score:      {f1_s:.4f}")
print(f"Precision:     {precision_s:.4f}")
print(f"Recall:        {recall_s:.4f}")
print(f"Specificity:   {specificity_s:.4f}")
print(f"Accuracy:      {accuracy_s:.4f}")

print("\nConfusion Matrix:")
print(cm_s)

S-XGBOOST PERFORMANCE
ROC-AUC:       0.7681
PR-AUC:        0.7395
F1-score:      0.6476
Precision:     0.6939
Recall:        0.6071
Specificity:   0.7541
Accuracy:      0.6838

Confusion Matrix:
[[46 15]
 [22 34]]


In [10]:
# ============================================================
# CELL 11- CLASSIFICATION REPORT
# ============================================================

print("=" * 70)
print("S-XGBOOST CLASSIFICATION REPORT")
print("=" * 70)

print(
    classification_report(
        y_test,
        y_pred_s,
        target_names=[
            "Not-At-Risk",
            "At-Risk"
        ],
        zero_division=0
    )
)

S-XGBOOST CLASSIFICATION REPORT
              precision    recall  f1-score   support

 Not-At-Risk       0.68      0.75      0.71        61
     At-Risk       0.69      0.61      0.65        56

    accuracy                           0.68       117
   macro avg       0.69      0.68      0.68       117
weighted avg       0.68      0.68      0.68       117



In [11]:
#Cell 12 — Compare Baseline vs S-XGBoostC--KEY ABLATION RESULTS
# ============================================================
# BASELINE VS S-XGBOOST
# ============================================================

baseline_results = pd.DataFrame({
    "Model": ["Baseline XGBoost"],
    "Features": [61],
    "ROC_AUC": [0.7567],
    "PR_AUC": [0.7220],
    "F1": [0.6296],
    "Precision": [0.6538],
    "Recall": [0.6071],
    "Specificity": [0.7049],
    "Accuracy": [0.6581]
})

s_results = pd.DataFrame({
    "Model": ["S-XGBoost"],
    "Features": [49],
    "ROC_AUC": [roc_auc_s],
    "PR_AUC": [pr_auc_s],
    "F1": [f1_s],
    "Precision": [precision_s],
    "Recall": [recall_s],
    "Specificity": [specificity_s],
    "Accuracy": [accuracy_s]
})

comparison = pd.concat(
    [baseline_results, s_results],
    ignore_index=True
)

print("=" * 70)
print("BASELINE vs S-XGBOOST")
print("=" * 70)

print(
    comparison.to_string(index=False)
)

BASELINE vs S-XGBOOST
           Model  Features  ROC_AUC   PR_AUC       F1  Precision   Recall  Specificity  Accuracy
Baseline XGBoost        61  0.75670 0.722000 0.629600   0.653800 0.607100     0.704900  0.658100
       S-XGBoost        49  0.76815 0.739462 0.647619   0.693878 0.607143     0.754098  0.683761


In [12]:
#Cell 13 — Calculate changes from baseline
# ============================================================
# S-XGBOOST IMPROVEMENT OVER BASELINE
# ============================================================

delta = {
    "ROC_AUC": roc_auc_s - 0.7567,
    "PR_AUC": pr_auc_s - 0.7220,
    "F1": f1_s - 0.6296,
    "Precision": precision_s - 0.6538,
    "Recall": recall_s - 0.6071,
    "Specificity": specificity_s - 0.7049,
    "Accuracy": accuracy_s - 0.6581
}

print("=" * 70)
print("CHANGE FROM BASELINE")
print("=" * 70)

for metric, change in delta.items():
    print(f"{metric:15s}: {change:+.4f}")

CHANGE FROM BASELINE
ROC_AUC        : +0.0114
PR_AUC         : +0.0175
F1             : +0.0180
Precision      : +0.0401
Recall         : +0.0000
Specificity    : +0.0492
Accuracy       : +0.0257


In [13]:
#Cell 14 — Save S-XGBoost mode
# ============================================================
# SAVE S-XGBOOST MODEL
# ============================================================

s_model_path = (
    MODELS_DIR / "s_xgboost_binary.pkl"
)

joblib.dump(
    s_xgboost,
    s_model_path
)

print("=" * 70)
print("S-XGBOOST MODEL SAVED")
print("=" * 70)

print(f"Model: {s_model_path}")

assert s_model_path.exists()

print("✓ S-XGBoost model successfully saved")

S-XGBOOST MODEL SAVED
Model: ..\models\s_xgboost_binary.pkl
✓ S-XGBoost model successfully saved


In [14]:
# ============================================================
# CELL 15 -SAVE S-XGBOOST RESULTS
# ============================================================

s_results.to_csv(
    RESULTS_DIR / "s_xgboost_binary_results.csv",
    index=False
)

comparison.to_csv(
    RESULTS_DIR / "baseline_vs_s_xgboost.csv",
    index=False
)

print("=" * 70)
print("S-XGBOOST RESULTS SAVED")
print("=" * 70)

print("✓ s_xgboost_binary_results.csv")
print("✓ baseline_vs_s_xgboost.csv")

S-XGBOOST RESULTS SAVED
✓ s_xgboost_binary_results.csv
✓ baseline_vs_s_xgboost.csv


In [15]:
# ============================================================
# CELL 16 --NOTEBOOK 6 — FINAL VALIDATION
# ============================================================

print("=" * 70)
print("NOTEBOOK 6 — FINAL VALIDATION")
print("=" * 70)

assert X_train_shap.shape == (468, 49)
assert X_test_shap.shape == (117, 49)

assert len(y_pred_s) == 117
assert len(y_prob_s) == 117

assert np.all(
    (y_prob_s >= 0) &
    (y_prob_s <= 1)
)

assert np.isfinite(roc_auc_s)
assert np.isfinite(pr_auc_s)
assert np.isfinite(f1_s)
assert np.isfinite(precision_s)
assert np.isfinite(recall_s)
assert np.isfinite(specificity_s)
assert np.isfinite(accuracy_s)

assert cm_s.shape == (2, 2)

assert s_model_path.exists()

print("✓ SHAP-selected predictors: 49")
print("✓ Training data: 468 × 49")
print("✓ Test data: 117 × 49")
print("✓ Predictions: 117")
print("✓ Probabilities valid")
print("✓ All metrics finite")
print("✓ Confusion matrix valid")
print("✓ S-XGBoost model saved")
print("✓ Test set remains 117 observations")

print("\n" + "=" * 70)
print("✓ NOTEBOOK 6 S-XGBOOST VALIDATION PASSED")
print("=" * 70)

NOTEBOOK 6 — FINAL VALIDATION
✓ SHAP-selected predictors: 49
✓ Training data: 468 × 49
✓ Test data: 117 × 49
✓ Predictions: 117
✓ Probabilities valid
✓ All metrics finite
✓ Confusion matrix valid
✓ S-XGBoost model saved
✓ Test set remains 117 observations

✓ NOTEBOOK 6 S-XGBOOST VALIDATION PASSED
